In [ ]:
# 1. Install (first run only). If Colab asks to restart the runtime, restart and continue from cell 2.
!pip install -q faster-whisper jiwer num2words "av<14" nvidia-cublas-cu12 nvidia-cudnn-cu12
# Optional second ASR (cell 17). A failure here does NOT affect the core pipeline.
!pip install -q "onnx-asr[cpu,hub]" "onnxruntime!=1.24.1" || echo "optional ASR2 install failed - core pipeline unaffected"
!nvidia-smi -L
!ffmpeg -version | head -1

/bin/bash: line 1: nvidia-smi: command not found
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


In [ ]:
# 2. Setup: folders, optional Drive, GPU libraries
import os, sys, glob, ctypes, site

PROJECT = "/content/meeting-assistant"
for d in ("stages", "eval", "outputs"):
    os.makedirs(f"{PROJECT}/{d}", exist_ok=True)
for init in (f"{PROJECT}/stages/__init__.py", f"{PROJECT}/eval/__init__.py"):
    open(init, "a").close()
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

# Optional Google Drive: keeps the 3 GB model cache and your test clips between sessions.
USE_DRIVE = True
CLIPS_DIR = f"{PROJECT}/eval/clips"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
        CLIPS_DIR = "/content/drive/MyDrive/meeting_assistant/clips"
    except Exception as e:
        print("Drive not mounted, using local folders:", str(e)[:100])
os.makedirs(CLIPS_DIR, exist_ok=True)

# GPU libraries: make the pip-installed cuBLAS/cuDNN visible to CTranslate2 inside THIS process.
def preload_cuda_libs():
    roots = list(dict.fromkeys(site.getsitepackages() + [site.getusersitepackages()]))
    dirs, loaded = [], []
    for r in roots:
        for sub in ("cublas", "cudnn"):
            d = os.path.join(r, "nvidia", sub, "lib")
            if os.path.isdir(d):
                dirs.append(d)
                for pat in ("libcublasLt.so*", "libcublas.so*", "libcudnn.so*"):
                    for p in sorted(glob.glob(os.path.join(d, pat))):
                        try:
                            ctypes.CDLL(p, mode=ctypes.RTLD_GLOBAL)
                            loaded.append(os.path.basename(p))
                        except OSError:
                            pass
    if dirs:   # also helps any subprocess you launch from the notebook
        os.environ["LD_LIBRARY_PATH"] = ":".join(dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
    return loaded

print("preloaded GPU libs:", preload_cuda_libs())
try:
    import ctranslate2
    print("GPUs visible to CTranslate2:", ctranslate2.get_cuda_device_count())
except Exception as e:
    print("ctranslate2 import problem:", e)
print("clips folder:", CLIPS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
preloaded GPU libs: ['libcublasLt.so.12', 'libcublas.so.12', 'libcudnn.so.9']
GPUs visible to CTranslate2: 0
clips folder: /content/drive/MyDrive/meeting_assistant/clips


In [ ]:
%%writefile config.py
import os
from dataclasses import dataclass, replace

SR = 16000


def _has_cuda() -> bool:
    try:
        import ctranslate2
        return ctranslate2.get_cuda_device_count() > 0
    except Exception:
        return False


_CUDA = _has_cuda()


@dataclass
class STTConfig:
    # ---- model ----
    model_size: str = os.getenv("WHISPER_MODEL", "large-v3")
    device: str = os.getenv("WHISPER_DEVICE", "cuda" if _CUDA else "cpu")
    compute_type: str = os.getenv("WHISPER_COMPUTE", "float16" if _CUDA else "int8")
    language: str = "en"
    beam_size: int = 5
    temperature: tuple = (0.0, 0.2, 0.4)
    # ---- hallucination firewall ----
    firewall: bool = True                 # False = "bare Whisper" baseline for ablations
    vad_threshold: float = 0.5
    vad_min_silence_ms: int = 500
    vad_speech_pad_ms: int = 400
    no_speech_drop: float = 0.6
    logprob_drop: float = -1.0
    compression_drop: float = 2.4
    repetition_penalty: float = 1.0       # 1.0 = off
    no_repeat_ngram_size: int = 0         # 0 = off
    hallucination_silence_threshold: float = 0.0   # seconds; 0 = off
    consistency: bool = False             # L3 self-consistency re-decode (2x time)
    consistency_min_similarity: float = 0.6
    # ---- vocabulary biasing ----
    prompt_style: str = "sentence"        # "sentence" | "list" | "none"
    prompt_max_chars: int = 400
    use_hotwords: bool = False
    # ---- confidence flags ----
    low_conf_word_prob: float = 0.5
    critical_prob: float = 0.8
    # ---- ingest ----
    max_duration_s: int = 3 * 3600
    max_file_mb: int = 500
    min_speech_s: float = 1.5
    ffmpeg_filter: str = ""               # e.g. "highpass=f=80"; empty = none (recommended default)
    # ---- optional second ASR (L4 cross-check) ----
    use_asr2: bool = True
    asr2_model: str = "nemo-parakeet-tdt-0.6b-v2"
    asr2_quantization: str = "int8"
    asr2_max_chunk_s: float = 25.0


CFG = STTConfig()


def with_changes(**kw) -> STTConfig:
    """Copy of the default config with some fields changed (used by the comparison grid)."""
    return replace(CFG, **kw)

Overwriting config.py


In [ ]:
%%writefile errors.py
class PipelineError(Exception):
    code = "pipeline_error"

    def __init__(self, user_message: str):
        super().__init__(user_message)
        self.user_message = user_message


class UnsupportedFormatError(PipelineError):
    code = "unsupported_format"


class EmptyAudioError(PipelineError):
    code = "empty_audio"


class DecodeError(PipelineError):
    code = "decode_error"


class NoSpeechError(PipelineError):
    code = "no_speech"


class TooLargeError(PipelineError):
    code = "too_large"


class ModelError(PipelineError):
    code = "model_error"

Overwriting errors.py


In [ ]:
%%writefile stages/textutil.py
import re

try:
    from num2words import num2words as _n2w
except Exception:       # works without num2words, just with weaker number normalisation
    _n2w = None

NEGATIONS = {"not", "no", "never", "cannot", "neither", "nor", "without", "none", "nobody", "nothing"}
COMMIT = {"will", "shall", "should", "might", "must", "maybe", "can", "could", "would"}
NUMBER_WORDS = set("""zero one two three four five six seven eight nine ten eleven twelve thirteen
fourteen fifteen sixteen seventeen eighteen nineteen twenty thirty forty fifty sixty seventy eighty
ninety hundred thousand million billion first second third fourth fifth sixth seventh eighth ninth
tenth twentieth thirtieth""".split())


def _digits_to_words(s: str) -> str:
    s = re.sub(r"(?<=\d),(?=\d{3})", "", s)            # 1,000 -> 1000
    s = re.sub(r"(?<=\d)\.(?=\d)", " point ", s)        # 3.5 -> 3 point 5
    if _n2w is None:
        return s

    def ordinal(m):
        try:
            return " " + _n2w(int(m.group(1)), to="ordinal") + " "
        except Exception:
            return m.group(0)

    def cardinal(m):
        try:
            return " " + _n2w(int(m.group())) + " "
        except Exception:
            return m.group(0)

    s = re.sub(r"(\d+)(st|nd|rd|th)\b", ordinal, s)
    s = re.sub(r"\d+", cardinal, s)
    return s


def norm_text(s: str) -> str:
    """Lowercase, numbers->words, no punctuation. Used for WER, term checks and alignment."""
    s = s.lower().replace("\u2019", "'")
    s = s.replace("%", " percent ").replace("&", " and ").replace("/", " ")
    s = _digits_to_words(s)
    s = s.replace("-", " ")
    s = re.sub(r"[^a-z0-9']", " ", s)
    return " ".join(s.split())


def is_critical(word: str) -> bool:
    """Numbers, negations and commitment words: the tokens the rubric cares most about."""
    w = re.sub(r"[^a-z0-9']", "", word.lower())
    if not w:
        return False
    return (w in NEGATIONS or w in COMMIT or w in NUMBER_WORDS
            or w.endswith("n't") or any(ch.isdigit() for ch in w))

Overwriting stages/textutil.py


In [ ]:
%%writefile stages/ingest.py
import inspect
import json
import os
import subprocess
import tempfile
import wave

import numpy as np

from config import CFG, SR
from errors import (DecodeError, EmptyAudioError, NoSpeechError,
                    TooLargeError, UnsupportedFormatError)

ALLOWED = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".webm",
           ".mp4", ".aac", ".wma", ".mkv"}


def _run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True)


def probe(path: str) -> dict:
    r = _run(["ffprobe", "-v", "error", "-show_entries",
              "format=duration:stream=codec_type", "-of", "json", path])
    if r.returncode != 0:
        raise DecodeError("This file could not be read as audio. It may be corrupted or not an audio file.")
    try:
        info = json.loads(r.stdout or "{}")
    except Exception:
        raise DecodeError("This file could not be read as audio.")
    if not any(s.get("codec_type") == "audio" for s in info.get("streams", [])):
        raise DecodeError("No audio track was found in this file.")
    try:
        dur = float(info.get("format", {}).get("duration") or 0)
    except (TypeError, ValueError):
        dur = 0.0
    return {"duration_s": dur}


def convert_to_wav(path: str, ffmpeg_filter: str = "") -> str:
    """16 kHz mono 16-bit PCM. No loudness normalisation by default (it can amplify noise)."""
    out = os.path.join(tempfile.mkdtemp(), "audio_16k_mono.wav")
    cmd = ["ffmpeg", "-y", "-i", path, "-vn", "-ac", "1", "-ar", str(SR)]
    if ffmpeg_filter:
        cmd += ["-af", ffmpeg_filter]
    cmd += ["-c:a", "pcm_s16le", out]
    r = _run(cmd)
    if r.returncode != 0 or not os.path.exists(out) or os.path.getsize(out) < 1000:
        raise DecodeError("Audio conversion failed. The file may be damaged.")
    return out


def load_wav(wav_path: str) -> np.ndarray:
    """Read 16-bit PCM WAV into float32 [-1, 1] (no PyAV needed)."""
    try:
        with wave.open(wav_path, "rb") as wf:
            if wf.getsampwidth() != 2:
                raise DecodeError("Unexpected audio format after conversion.")
            frames = wf.readframes(wf.getnframes())
            audio = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0
            if wf.getnchannels() > 1:
                audio = audio.reshape(-1, wf.getnchannels()).mean(axis=1)
            return audio
    except DecodeError:
        raise
    except Exception:
        raise DecodeError("Could not read the converted audio.")


def _energy_segments(audio: np.ndarray, cfg=CFG):
    """Adaptive energy fallback if Silero VAD is unavailable. Threshold follows the noise floor,
    so a constant tone or steady noise is NOT mistaken for speech."""
    frame = int(0.03 * SR)
    n = len(audio) // frame
    if n == 0:
        return []
    rms = np.sqrt((audio[: n * frame].reshape(n, frame) ** 2).mean(axis=1))
    thr = max(0.002, 3.0 * float(np.percentile(rms, 10)))
    mask = rms > thr
    segs, start = [], None
    for i, m in enumerate(mask):
        if m and start is None:
            start = i
        elif not m and start is not None:
            segs.append([start * 0.03, i * 0.03])
            start = None
    if start is not None:
        segs.append([start * 0.03, n * 0.03])
    merged = []
    gap = cfg.vad_min_silence_ms / 1000.0
    for s, e in segs:
        if merged and s - merged[-1][1] <= gap:
            merged[-1][1] = e
        else:
            merged.append([s, e])
    pad = cfg.vad_speech_pad_ms / 1000.0
    total = len(audio) / SR
    return [(max(0.0, s - pad), min(total, e + pad)) for s, e in merged if e - s >= 0.2]


def vad_segments(audio: np.ndarray, cfg=CFG):
    """Returns ([(start_s, end_s), ...], method). Silero VAD (bundled with faster-whisper) with fallback."""
    total_s = len(audio) / SR
    if total_s == 0:
        return [], "none"
    try:
        from faster_whisper.vad import VadOptions, get_speech_timestamps
        params = dict(threshold=cfg.vad_threshold,
                      min_silence_duration_ms=cfg.vad_min_silence_ms,
                      speech_pad_ms=cfg.vad_speech_pad_ms)
        allowed = set(inspect.signature(VadOptions).parameters)
        opts = VadOptions(**{k: v for k, v in params.items() if k in allowed})
        try:
            ts = get_speech_timestamps(audio, opts, sampling_rate=SR)
        except TypeError:
            ts = get_speech_timestamps(audio, opts)
        if not ts:
            return [], "silero"
        scale = 1.0 if ts[-1]["end"] <= total_s + 1.0 else 1.0 / SR   # seconds or samples?
        return [(t["start"] * scale, t["end"] * scale) for t in ts], "silero"
    except Exception:
        return _energy_segments(audio, cfg), "energy"


def validate_and_convert(path: str, cfg=CFG) -> dict:
    if not path or not os.path.exists(path):
        raise EmptyAudioError("No file was provided.")
    size = os.path.getsize(path)
    if size == 0:
        raise EmptyAudioError("The uploaded file is empty (0 bytes).")
    if size > cfg.max_file_mb * 1024 * 1024:
        raise TooLargeError(f"File is larger than {cfg.max_file_mb} MB.")
    ext = os.path.splitext(path)[1].lower()
    if ext not in ALLOWED:
        raise UnsupportedFormatError(
            f"Unsupported file type '{ext}'. Supported: {', '.join(sorted(ALLOWED))}.")
    meta = probe(path)
    if meta["duration_s"] < 0.5:
        raise EmptyAudioError("The recording is empty or too short to process.")
    if meta["duration_s"] > cfg.max_duration_s:
        raise TooLargeError(f"Recording is longer than {cfg.max_duration_s // 3600} hours.")
    wav = convert_to_wav(path, cfg.ffmpeg_filter)
    audio = load_wav(wav)
    segs, method = vad_segments(audio, cfg)
    speech_s = sum(e - s for s, e in segs)
    if speech_s < cfg.min_speech_s:
        raise NoSpeechError("No speech was detected in this recording. "
                            "It may be silent or contain only noise or music.")
    return {"wav_path": wav, "audio": audio, "duration_s": meta["duration_s"],
            "speech_s": round(speech_s, 1), "speech_segments": segs, "vad_method": method}

Overwriting stages/ingest.py


In [ ]:
%%writefile stages/stt.py
import difflib
import gc
import inspect
from dataclasses import asdict
from functools import lru_cache

import numpy as np

from config import CFG, SR
from errors import ModelError
from stages.textutil import norm_text, is_critical

PHANTOMS = [
    "thank you for watching", "thanks for watching", "please subscribe",
    "like and subscribe", "subtitles by", "see you in the next video",
    "transcribed by", "amara.org",
]


# ----------------------------------------------------------------- model handling
@lru_cache(maxsize=1)
def _load_model(model_size, device, compute_type):
    from faster_whisper import WhisperModel
    return WhisperModel(model_size, device=device, compute_type=compute_type)


def get_model(cfg=CFG):
    try:
        return _load_model(cfg.model_size, cfg.device, cfg.compute_type)
    except Exception as e:
        raise _model_error(e)


def free_gpu():
    """Release the Whisper model (call before loading an LLM so both fit on a 16 GB T4)."""
    _load_model.cache_clear()
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass


def _model_error(e) -> ModelError:
    msg = str(e)
    low = msg.lower()
    if any(k in low for k in ("cudnn", "cublas", "libcuda", "cuda")):
        return ModelError(
            "GPU libraries could not be loaded (" + msg[:140] + "). Re-run the 'GPU setup' cell, "
            "restart the runtime if you just installed packages, or use WHISPER_DEVICE=cpu WHISPER_MODEL=small.")
    return ModelError("Speech model failed: " + msg[:200])


def _filter_kwargs(fn, kwargs):
    """Drop arguments this faster-whisper version does not know (protects against version drift)."""
    try:
        params = inspect.signature(fn).parameters
        if any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values()):
            return dict(kwargs), []
        ok = {k: v for k, v in kwargs.items() if k in params}
        return ok, sorted(set(kwargs) - set(ok))
    except (TypeError, ValueError):
        return dict(kwargs), []


# ----------------------------------------------------------------- firewall helpers
def has_loop(text: str) -> bool:
    toks = norm_text(text).split()
    for n, reps in ((1, 6), (2, 4), (3, 4), (4, 4)):
        for i in range(0, len(toks) - n * reps + 1):
            chunk = toks[i:i + n]
            if all(toks[i + k * n: i + (k + 1) * n] == chunk for k in range(reps)):
                return True
    return False


def drop_reason(seg: dict, prev_kept_text: str, dup_count: int, cfg=CFG):
    t = norm_text(seg["text"])
    if not t:
        return "empty"
    if seg["no_speech_prob"] > cfg.no_speech_drop and seg["avg_logprob"] < cfg.logprob_drop:
        return "no_speech_low_confidence"
    if seg["compression_ratio"] > cfg.compression_drop:
        return "high_compression_ratio(loop)"
    if has_loop(seg["text"]):
        return "repetition_loop"
    if len(t.split()) <= 12 and any(p in t for p in PHANTOMS) and \
            (seg["no_speech_prob"] > 0.2 or seg["avg_logprob"] < -0.7):
        return "phantom_phrase"
    if t == prev_kept_text and dup_count >= 2:
        return "duplicate_segment"
    return None


def detect_language_safe(model, audio, speech_segments=None):
    try:
        start = int((speech_segments[0][0] if speech_segments else 0.0) * SR)
        clip = audio[start:start + SR * 30]
        if len(clip) < SR:
            clip = audio[: SR * 30]
        res = model.detect_language(clip)
        return res[0], float(res[1])
    except Exception:
        return None, None


def build_prompt(glossary, cfg=CFG):
    if not glossary or cfg.prompt_style == "none":
        return None
    kept, used = [], 0
    for term in glossary:
        if used + len(term) + 2 > cfg.prompt_max_chars:
            break
        kept.append(term)
        used += len(term) + 2
    if not kept:
        return None
    terms = ", ".join(kept)
    if cfg.prompt_style == "list":
        return terms
    return f"This is a meeting transcript. Vocabulary: {terms}."


def build_kwargs(cfg, glossary):
    kw = dict(language=cfg.language, beam_size=cfg.beam_size, word_timestamps=True,
              initial_prompt=build_prompt(glossary, cfg))
    if cfg.use_hotwords and glossary:
        kw["hotwords"] = ", ".join(glossary)[: cfg.prompt_max_chars]
    if cfg.firewall:
        kw.update(
            vad_filter=True,
            vad_parameters=dict(threshold=cfg.vad_threshold,
                                min_silence_duration_ms=cfg.vad_min_silence_ms,
                                speech_pad_ms=cfg.vad_speech_pad_ms),
            condition_on_previous_text=False,
            temperature=list(cfg.temperature))
        if cfg.repetition_penalty and cfg.repetition_penalty != 1.0:
            kw["repetition_penalty"] = cfg.repetition_penalty
        if cfg.no_repeat_ngram_size:
            kw["no_repeat_ngram_size"] = cfg.no_repeat_ngram_size
        if cfg.hallucination_silence_threshold and cfg.hallucination_silence_threshold > 0:
            kw["hallucination_silence_threshold"] = cfg.hallucination_silence_threshold
    else:   # "bare Whisper": what most people run by default
        kw.update(vad_filter=False, condition_on_previous_text=True)
    return kw


# ----------------------------------------------------------------- L3 self-consistency
def _annotate_stability(model, audio, kept, kwargs, cfg):
    kw = dict(kwargs)
    kw.update(beam_size=1, temperature=[0.0], initial_prompt=None, condition_on_previous_text=False)
    for k in ("hotwords", "hallucination_silence_threshold", "repetition_penalty", "no_repeat_ngram_size"):
        kw.pop(k, None)
    kw["word_timestamps"] = False
    kw, _ = _filter_kwargs(model.transcribe, kw)
    it, _info = model.transcribe(audio, **kw)
    second = [(s.start, s.end, norm_text(s.text)) for s in it]
    for seg in kept:
        a = norm_text(seg["text"]).split()
        overlap = []
        for s0, s1, txt in second:
            ov = min(seg["end"], s1) - max(seg["start"], s0)
            if ov > 0.3 * max(0.1, min(seg["end"] - seg["start"], s1 - s0)):
                overlap.append(txt)
        b = " ".join(overlap).split()
        ratio = difflib.SequenceMatcher(None, a, b, autojunk=False).ratio() if (a and b) else 0.0
        seg["stability"] = round(ratio, 2)
        seg["unstable"] = ratio < cfg.consistency_min_similarity


# ----------------------------------------------------------------- main entry
def transcribe(audio, glossary=None, cfg=None, on_progress=None, speech_segments=None) -> dict:
    """audio: float32 mono 16 kHz numpy array. Returns the raw-transcript dict."""
    cfg = cfg or CFG
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim != 1 or audio.size == 0:
        raise ModelError("Internal error: audio must be a non-empty mono array.")
    total_s = len(audio) / SR
    model = get_model(cfg)

    warnings = []
    lang, lang_p = detect_language_safe(model, audio, speech_segments)
    if lang and lang != cfg.language and lang_p and lang_p > 0.7:
        warnings.append(f"Audio may not be '{cfg.language}' (detected '{lang}', p={lang_p:.2f}). "
                        "Results may be inaccurate.")

    kwargs = build_kwargs(cfg, glossary)
    kwargs, ignored = _filter_kwargs(model.transcribe, kwargs)
    if ignored:
        warnings.append("Ignored options not supported by this faster-whisper version: " + ", ".join(ignored))

    kept, dropped = [], []
    prev_text, dup = "", 0
    try:
        seg_iter, info = model.transcribe(audio, **kwargs)
        for s in seg_iter:
            words = [{"w": w.word, "start": round(w.start, 2), "end": round(w.end, 2),
                      "p": round(w.probability, 3)} for w in (s.words or [])]
            seg = {"id": len(kept) + len(dropped) + 1,
                   "start": round(s.start, 2), "end": round(s.end, 2), "text": s.text.strip(),
                   "avg_logprob": round(s.avg_logprob, 3), "no_speech_prob": round(s.no_speech_prob, 3),
                   "compression_ratio": round(s.compression_ratio, 2), "words": words}
            if on_progress and total_s > 0:
                on_progress(min(1.0, s.end / total_s))
            reason = drop_reason(seg, prev_text, dup, cfg) if cfg.firewall else None
            if reason:
                seg["drop_reason"] = reason
                dropped.append(seg)
                continue
            norm_seg = norm_text(seg["text"])
            dup = dup + 1 if norm_seg == prev_text else 0
            prev_text = norm_seg
            seg["low_conf_words"] = [w["w"].strip() for w in words if w["p"] < cfg.low_conf_word_prob]
            seg["critical_low_conf"] = [w["w"].strip() for w in words
                                        if is_critical(w["w"]) and w["p"] < cfg.critical_prob]
            kept.append(seg)
        if cfg.consistency and kept:
            _annotate_stability(model, audio, kept, kwargs, cfg)
    except RuntimeError as e:
        raise _model_error(e)

    if not kept:
        warnings.append("No transcript text was produced after filtering.")
    return {
        "text": " ".join(s["text"] for s in kept).strip(),
        "segments": kept,
        "dropped": dropped,
        "flags": {"low_conf_words": sum(len(s["low_conf_words"]) for s in kept),
                  "critical_low_conf": sum(len(s["critical_low_conf"]) for s in kept),
                  "dropped_segments": len(dropped),
                  "unstable_segments": sum(1 for s in kept if s.get("unstable"))},
        "info": {"model": f"faster-whisper/{cfg.model_size}", "device": cfg.device,
                 "compute_type": cfg.compute_type, "language": info.language,
                 "language_probability": round(float(info.language_probability), 3),
                 "duration": round(total_s, 1),
                 "duration_after_vad": round(float(getattr(info, "duration_after_vad", total_s)), 1),
                 "firewall": cfg.firewall, "glossary_used": bool(glossary),
                 "prompt": kwargs.get("initial_prompt"), "config": asdict(cfg)},
        "warnings": warnings,
    }

Overwriting stages/stt.py


In [ ]:
%%writefile stages/asr2.py
"""Optional L4 cross-check: a second, architecturally different ASR (NVIDIA Parakeet TDT via onnx-asr).
It never replaces the Whisper transcript. It only marks WHERE the two systems disagree, so Phase 2
can focus on (and be cautious with) those spans.
Install (CPU, no NeMo / PyTorch needed):  pip install "onnx-asr[cpu,hub]" "onnxruntime!=1.24.1"
"""
import difflib
import time
from collections import Counter

import numpy as np

from config import CFG, SR
from stages.textutil import norm_text, is_critical

_MODELS = {}


def load(cfg=CFG):
    import onnx_asr
    key = (cfg.asr2_model, cfg.asr2_quantization)
    if key not in _MODELS:
        try:
            if cfg.asr2_quantization:
                _MODELS[key] = onnx_asr.load_model(cfg.asr2_model, quantization=cfg.asr2_quantization)
            else:
                _MODELS[key] = onnx_asr.load_model(cfg.asr2_model)
        except Exception:
            _MODELS[key] = onnx_asr.load_model(cfg.asr2_model)     # retry without quantisation
    return _MODELS[key]


def chunk_plan(speech_segments, total_s, max_chunk_s=25.0, max_gap_s=1.0, pad=0.15):
    segs = list(speech_segments) if speech_segments else [(0.0, total_s)]
    chunks, cur = [], None
    for s, e in segs:
        pieces, t = [], s
        while e - t > max_chunk_s:
            pieces.append((t, t + max_chunk_s))
            t += max_chunk_s
        pieces.append((t, e))
        for ps, pe in pieces:
            if cur and ps - cur[1] <= max_gap_s and pe - cur[0] <= max_chunk_s:
                cur = (cur[0], pe)
            else:
                if cur:
                    chunks.append(cur)
                cur = (ps, pe)
    if cur:
        chunks.append(cur)
    return [(max(0.0, s - pad), min(total_s, e + pad)) for s, e in chunks if e - s >= 0.3]


def transcribe_chunks(audio, speech_segments, cfg=CFG):
    model = load(cfg)
    total_s = len(audio) / SR
    out = []
    for s, e in chunk_plan(speech_segments, total_s, cfg.asr2_max_chunk_s):
        clip = np.ascontiguousarray(audio[int(s * SR): int(e * SR)], dtype=np.float32)
        try:
            res = model.recognize(clip, sample_rate=SR)
            if isinstance(res, (list, tuple)):
                res = res[0] if res else ""
            text = res if isinstance(res, str) else getattr(res, "text", str(res))
        except Exception as ex:
            text = ""
            out.append({"start": round(s, 2), "end": round(e, 2), "text": "", "error": str(ex)[:120]})
            continue
        out.append({"start": round(s, 2), "end": round(e, 2), "text": text.strip()})
    return out


def _whisper_tokens(raw):
    toks = []
    for seg in raw["segments"]:
        added = False
        for w in seg.get("words", []):
            for t in norm_text(w["w"]).split():
                toks.append((t, seg["id"], w["start"], w["end"]))
                added = True
        if not added:
            for t in norm_text(seg["text"]).split():
                toks.append((t, seg["id"], seg["start"], seg["end"]))
    return toks


def compare(raw, chunks, model_name=None):
    wt = _whisper_tokens(raw)
    pt = []
    for c in chunks:
        for t in norm_text(c["text"]).split():
            pt.append((t, c["start"], c["end"]))
    a = [x[0] for x in wt]
    b = [x[0] for x in pt]
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    flagged, dis = set(), []
    seg_bad = Counter()
    seg_n = Counter(x[1] for x in wt)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        if tag == "insert":
            idx = [k for k in (i1 - 1, i1) if 0 <= k < len(a)]
        else:
            idx = list(range(i1, i2))
        flagged.update(idx)
        for k in range(i1, i2):
            seg_bad[wt[k][1]] += 1
        if wt and i2 > i1:
            start, end = wt[i1][2], wt[i2 - 1][3]
            seg_ids = sorted({wt[k][1] for k in range(i1, i2)})
        elif wt:
            ref_tok = wt[min(max(i1 - 1, 0), len(wt) - 1)]
            start, end, seg_ids = ref_tok[2], ref_tok[3], [ref_tok[1]]
        else:
            start, end, seg_ids = (pt[j1][1], pt[j1][2], []) if pt else (0.0, 0.0, [])
        wtxt, ptxt = " ".join(a[i1:i2]), " ".join(b[j1:j2])
        dis.append({"id": len(dis) + 1, "type": tag, "whisper": wtxt, "parakeet": ptxt,
                    "start": round(start, 2), "end": round(end, 2), "seg_ids": seg_ids,
                    "critical": any(is_critical(t) for t in (a[i1:i2] + b[j1:j2])),
                    "context": " ".join(a[max(0, i1 - 6): i2 + 6])})
    for seg in raw["segments"]:
        n, bad = seg_n.get(seg["id"], 0), seg_bad.get(seg["id"], 0)
        seg["asr2_diff_frac"] = round(bad / n, 2) if n else 0.0
        seg["asr2_unsupported"] = bool(n >= 3 and bad / n >= 0.8)
    return {"model": model_name or CFG.asr2_model, "text": " ".join(c["text"] for c in chunks).strip(),
            "chunks": chunks, "disagreements": dis,
            "disagreement_rate": round(len(flagged) / max(len(a), 1), 3),
            "critical_disagreements": sum(1 for d in dis if d["critical"]),
            "unsupported_segments": sum(1 for s in raw["segments"] if s.get("asr2_unsupported")),
            "flagged_token_idx": sorted(flagged), "whisper_tokens": a}


def cross_check(raw, audio, speech_segments, cfg=CFG):
    t0 = time.time()
    chunks = transcribe_chunks(audio, speech_segments, cfg)
    res = compare(raw, chunks, cfg.asr2_model)
    res["seconds"] = round(time.time() - t0, 1)
    return res

Overwriting stages/asr2.py


In [ ]:
%%writefile pipeline_stt.py
import json
import os
import sys

from config import CFG
from errors import PipelineError
from stages import ingest, stt


def _fmt(sec: float) -> str:
    sec = int(sec)
    h, rem = divmod(sec, 3600)
    m, s = divmod(rem, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"


def to_numbered_lines(raw: dict) -> list:
    """['[L001 00:03] text', ...] -- the format Phase 3 (documenter) will cite as evidence."""
    return [f"[L{i:03d} {_fmt(s['start'])}] {s['text']}" for i, s in enumerate(raw["segments"], 1)]


def process(audio_path: str, glossary=None, cfg=None, on_progress=None) -> dict:
    cfg = cfg or CFG
    report = on_progress or (lambda stage, p: None)
    report("validating", 0.02)
    meta = ingest.validate_and_convert(audio_path, cfg)
    report("transcribing", 0.05)
    raw = stt.transcribe(meta["audio"], glossary=glossary, cfg=cfg,
                         on_progress=lambda f: report("transcribing", 0.05 + 0.80 * f),
                         speech_segments=meta["speech_segments"])
    raw["info"].update({"input_duration_s": meta["duration_s"], "speech_s": meta["speech_s"],
                        "vad_method": meta["vad_method"]})
    if cfg.use_asr2:                       # optional, never fatal
        report("cross-checking", 0.88)
        try:
            from stages import asr2
            raw["asr2"] = asr2.cross_check(raw, meta["audio"], meta["speech_segments"], cfg)
        except Exception as e:
            raw["warnings"].append("Second-ASR cross-check skipped: " + str(e)[:160])
    raw["lines"] = to_numbered_lines(raw)
    report("done", 1.0)
    return raw


def save_outputs(raw: dict, out_dir: str = "outputs", audio_path: str = None, glossary=None) -> dict:
    os.makedirs(out_dir, exist_ok=True)
    jp, tp = os.path.join(out_dir, "raw_transcript.json"), os.path.join(out_dir, "raw_transcript.txt")
    with open(jp, "w", encoding="utf-8") as f:
        json.dump(raw, f, indent=2, ensure_ascii=False)
    with open(tp, "w", encoding="utf-8") as f:
        f.write("\n".join(raw["lines"]))

    # Bundle contract record with Parakeet findings
    rec = {
        "schema_version": "1.0",
        "meta": {
            "audio_file": os.path.basename(audio_path) if audio_path else None,
            "models": {"stt": raw.get("info", {}).get("model")},
            "flags": raw.get("flags", {})
        },
        "raw_transcript": {
            "text": raw["text"],
            "segments": raw["segments"],
            "lines": raw.get("lines", to_numbered_lines(raw))
        },
        "low_confidence_words": [
            {"segment_id": s["id"], "word": w["w"].strip(), "p": w.get("p")}
            for s in raw["segments"] for w in s.get("words", [])
            if w.get("w", "").strip() in s.get("low_conf_words", [])
        ],
        "dropped_segments": raw.get("dropped", []),
        "asr2": raw.get("asr2"),
        "glossary": list(glossary or [])
    }
    rp = os.path.join(out_dir, "stage1_record.json")
    with open(rp, "w", encoding="utf-8") as f:
        json.dump(rec, f, indent=2, ensure_ascii=False)
    return {"json": jp, "txt": tp, "record": rp}


if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python pipeline_stt.py <audio_path> [glossary terms ...]")
        sys.exit(1)
    try:
        out = process(sys.argv[1], glossary=sys.argv[2:] or None)
        print(out["text"])
        print("\nDROPPED:", [(d["text"], d["drop_reason"]) for d in out["dropped"]])
        print(save_outputs(out))
    except PipelineError as e:
        print(f"[{e.code}] {e.user_message}")

Overwriting pipeline_stt.py


In [ ]:
%%writefile eval/run_grid.py
import csv
import difflib
import os
import time
from collections import Counter

from config import CFG, with_changes
from stages.textutil import norm_text, is_critical

AUDIO_EXT = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".webm", ".mp4", ".aac"}

PLANTED = ["PostgreSQL", "MySQL", "CI/CD", "Kubernetes", "Docker", "API", "AWS", "Terraform",
           "runbook", "Priya", "Rahul", "fifteen", "fifty", "twenty third", "will not"]
ALIASES = {"MySQL": ["mysql", "my sql", "my sequel"],
           "PostgreSQL": ["postgresql", "postgres ql", "postgres sql", "postgre sql", "postgres"],
           "CI/CD": ["ci cd", "cicd", "c i c d", "ci slash cd"],
           "AWS": ["aws", "a w s"], "API": ["api", "a p i"], "runbook": ["runbook", "run book"]}
# Partial glossary on purpose: terms NOT in it show whether biasing helps or hurts the rest.
GLOSSARY = ["PostgreSQL", "Kubernetes", "Terraform", "Priya", "Rahul"]

CONFIGS = {   # name -> changes to the default config (glossary key = terms given to the decoder)
    "A bare whisper":      dict(firewall=False, prompt_style="none"),
    "B firewall (default)": dict(),
    "C +glossary sentence": dict(glossary=GLOSSARY),
    "D +glossary list":     dict(glossary=GLOSSARY, prompt_style="list"),
    "E +hotwords only":     dict(glossary=GLOSSARY, prompt_style="none", use_hotwords=True),
    "F beam=1":             dict(beam_size=1),
    "G no_repeat_ngram=4":  dict(no_repeat_ngram_size=4),
    "H vad_thr=0.35":       dict(vad_threshold=0.35),
    "I halluc_silence=2s":  dict(hallucination_silence_threshold=2.0),
    "J prep highpass80":    dict(ffmpeg_filter="highpass=f=80"),
    "K prep loudnorm":      dict(ffmpeg_filter="loudnorm"),
    "L +L3 consistency":    dict(consistency=True),
    "M turbo":              dict(model_size="large-v3-turbo"),   # keep LAST (model switch)
}


# ----------------------------------------------------------------- metrics
def wer(ref: str, hyp: str) -> float:
    r, h = norm_text(ref).split(), norm_text(hyp).split()
    if not r:
        return float(len(h) > 0)
    try:
        import jiwer
        return float(jiwer.wer(" ".join(r), " ".join(h) if h else ""))
    except Exception:
        prev = list(range(len(h) + 1))        # plain word-level edit distance
        for i in range(1, len(r) + 1):
            cur = [i] + [0] * len(h)
            for j in range(1, len(h) + 1):
                cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (r[i - 1] != h[j - 1]))
            prev = cur
        return prev[-1] / len(r)


def missing_terms(hyp: str, terms) -> list:
    """Whole-word match with spelling variants (so 'My SQL' counts as MySQL, 'rapid' is not 'API')."""
    h = f" {norm_text(hyp)} "
    out = []
    for t in terms:
        variants = [norm_text(t)] + [norm_text(v) for v in ALIASES.get(t, [])]
        if not any(f" {v} " in h for v in variants if v):
            out.append(t)
    return out


def critical_recall(ref: str, hyp: str):
    r = [t for t in norm_text(ref).split() if is_critical(t)]
    if not r:
        return None
    h = Counter(t for t in norm_text(hyp).split() if is_critical(t))
    hit = sum(min(c, h[t]) for t, c in Counter(r).items())
    return round(hit / len(r), 3)


def show_diff(ref: str, hyp: str, max_lines=40):
    r, h = norm_text(ref).split(), norm_text(hyp).split()
    n = 0
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, r, h, autojunk=False).get_opcodes():
        if tag != "equal":
            print(f"{tag:8} REF: {' '.join(r[i1:i2]) or '-':35} HYP: {' '.join(h[j1:j2]) or '-'}")
            n += 1
            if n >= max_lines:
                print("...")
                break
    if n == 0:
        print("No differences after normalisation.")


def error_capture(ref: str, raw: dict):
    """How well do Whisper-vs-Parakeet disagreements point at Whisper's real errors?"""
    a2 = raw.get("asr2")
    if not a2:
        return None
    r, a = norm_text(ref).split(), a2["whisper_tokens"]
    err = set()
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, r, a, autojunk=False).get_opcodes():
        if tag in ("replace", "insert"):
            err.update(range(j1, j2))
        elif tag == "delete":
            err.update(k for k in (j1 - 1, j1) if 0 <= k < len(a))
    flagged = set(a2["flagged_token_idx"])
    hit = len(err & flagged)
    return {"whisper_errors": len(err), "flagged": len(flagged), "hit": hit,
            "recall": round(hit / len(err), 3) if err else None,
            "precision": round(hit / len(flagged), 3) if flagged else None}


# ----------------------------------------------------------------- clips
def find_pairs(clips_dir: str):
    pairs = []
    if not os.path.isdir(clips_dir):
        return pairs
    for f in sorted(os.listdir(clips_dir)):
        stem, ext = os.path.splitext(f)
        ref = os.path.join(clips_dir, stem + ".txt")
        if ext.lower() in AUDIO_EXT and os.path.exists(ref):
            pairs.append((os.path.join(clips_dir, f), ref))
    return pairs


def load_terms(audio_path: str):
    stem = os.path.splitext(audio_path)[0]
    tf = stem + ".terms.txt"
    if os.path.exists(tf):
        return [l.strip() for l in open(tf, encoding="utf-8") if l.strip()]
    return PLANTED if os.path.basename(stem).lower().startswith("clipa") else []


# ----------------------------------------------------------------- grid
def run(pairs, configs=None, out_csv="outputs/grid_results.csv", include=None):
    """Compare configurations on (audio, reference) pairs. Everything runs in this one process."""
    from pipeline_stt import process
    from stages import stt
    configs = configs or CONFIGS
    if include:
        configs = {k: v for k, v in configs.items() if any(k.startswith(p) for p in include)}
    rows = []
    for audio, ref_path in pairs:
        ref = open(ref_path, encoding="utf-8").read()
        terms = load_terms(audio)
        for name, change in configs.items():
            change = dict(change)
            glossary = change.pop("glossary", None)
            t0 = time.time()
            row = {"clip": os.path.basename(audio), "config": name}
            try:
                out = process(audio, glossary=glossary, cfg=with_changes(**change))
                miss = missing_terms(out["text"], terms) if terms else []
                row.update({"WER": round(wer(ref, out["text"]), 4), "missing_terms": len(miss),
                            "missing": ";".join(miss), "crit_recall": critical_recall(ref, out["text"]),
                            "dropped": out["flags"]["dropped_segments"],
                            "crit_low_conf": out["flags"]["critical_low_conf"],
                            "unstable": out["flags"]["unstable_segments"],
                            "seconds": round(time.time() - t0, 1), "error": ""})
            except Exception as e:
                row.update({"WER": None, "error": f"{type(e).__name__}: {str(e)[:100]}",
                            "seconds": round(time.time() - t0, 1)})
            rows.append(row)
            print({k: row.get(k) for k in ("clip", "config", "WER", "missing_terms", "crit_recall", "dropped", "seconds", "error")})
    stt.free_gpu()
    os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
    keys = ["clip", "config", "WER", "missing_terms", "missing", "crit_recall", "dropped",
            "crit_low_conf", "unstable", "seconds", "error"]
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)
    print("saved", out_csv)
    return rows


def summarize(rows):
    """Per-config averages over clips (needs pandas, preinstalled on Colab)."""
    import pandas as pd
    df = pd.DataFrame(rows)
    ok = df[df["error"].fillna("") == ""]
    if ok.empty:
        return df
    g = ok.groupby("config", sort=False).agg(WER=("WER", "mean"), missing=("missing_terms", "sum"),
                                              crit_recall=("crit_recall", "mean"), dropped=("dropped", "sum"),
                                              seconds=("seconds", "mean")).round(4)
    return g


def recommend(summary, baseline="B firewall (default)", min_gain=0.01):
    """Keep the default unless another config clearly wins (>= 1 WER point) without losing critical-token recall."""
    if baseline not in summary.index:
        return baseline
    base = summary.loc[baseline]
    best, best_wer = baseline, base["WER"]
    for name, r in summary.iterrows():
        if name == baseline:
            continue
        if r["crit_recall"] < base["crit_recall"] - 1e-9:
            continue
        if base["WER"] - r["WER"] >= min_gain and r["WER"] < best_wer:
            best, best_wer = name, r["WER"]
    return best

Overwriting eval/run_grid.py


In [ ]:
%%writefile eval/hallucination.py
import os
import wave

import numpy as np

from config import SR, with_changes
from stages import ingest, stt


def _rms_scale(x, rms):
    cur = float(np.sqrt(np.mean(x ** 2))) or 1.0
    return (x * (rms / cur)).astype(np.float32)


def synth(kind: str, seconds: int = 30, seed: int = 0) -> np.ndarray:
    n = SR * seconds
    rng = np.random.default_rng(seed)
    if kind == "silence":
        return np.zeros(n, dtype=np.float32)
    white = rng.standard_normal(n)
    if kind == "white_noise":
        return _rms_scale(white, 0.05)
    if kind == "pink_noise":
        spec = np.fft.rfft(white)
        f = np.maximum(np.arange(len(spec)), 1)
        return _rms_scale(np.fft.irfft(spec / np.sqrt(f), n), 0.05)
    if kind == "brown_noise":
        b = np.cumsum(white)
        b -= np.linspace(b[0], b[-1], n)
        return _rms_scale(b, 0.05)
    t = np.arange(n) / SR
    if kind == "tones":       # simple chord with slow amplitude swell (stand-in for music)
        x = sum(np.sin(2 * np.pi * f * t) for f in (220, 277.2, 329.6)) * (0.6 + 0.4 * np.sin(2 * np.pi * 0.25 * t))
        return _rms_scale(x, 0.08)
    if kind == "hum":         # 50 Hz mains hum + harmonics + faint hiss
        x = sum(np.sin(2 * np.pi * 50 * k * t) / k for k in (1, 2, 3)) + 0.05 * white
        return _rms_scale(x, 0.05)
    raise ValueError(kind)


def write_wav(path: str, audio: np.ndarray):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    pcm = (np.clip(audio, -1, 1) * 32767).astype(np.int16)
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(SR)
        wf.writeframes(pcm.tobytes())


def make_clips(out_dir: str = "outputs/halluc_clips", speech_audio=None) -> dict:
    clips = {}
    for k in ("silence", "white_noise", "pink_noise", "brown_noise", "tones", "hum"):
        p = os.path.join(out_dir, f"{k}.wav")
        write_wav(p, synth(k))
        clips[k] = p
    if speech_audio is not None and len(speech_audio) > 4 * SR:
        mid = len(speech_audio) // 2
        gap = np.concatenate([speech_audio[:mid], synth("silence", 20), synth("pink_noise", 10), speech_audio[mid:]])
        p = os.path.join(out_dir, "speech_with_gap.wav")
        write_wav(p, gap)
        clips["speech_with_gap"] = p
        clips["_gap_interval"] = (mid / SR, mid / SR + 30.0)
    return clips


def _words_in(raw, lo, hi):
    n = 0
    for s in raw["segments"]:
        for w in s["words"]:
            if lo <= w["start"] <= hi:
                n += 1
    return n


def run(clips: dict, extra_wavs=()):
    """Compare bare Whisper with the full firewall on non-speech clips. Returns list of row dicts."""
    bare_cfg, fw_cfg = with_changes(firewall=False, prompt_style="none"), with_changes()
    gap = clips.get("_gap_interval")
    items = [(k, p) for k, p in clips.items() if not k.startswith("_")] + [(os.path.basename(p), p) for p in extra_wavs]
    rows = []
    print(f"{'clip':22}{'gate':>9}{'bare words':>12}{'firewall words':>16}{'gap words (bare/fw)':>22}")
    for name, path in items:
        audio = ingest.load_wav(path)
        segs, _ = ingest.vad_segments(audio, fw_cfg)
        sp = sum(e - s for s, e in segs)
        gate = "BLOCKED" if sp < fw_cfg.min_speech_s else "passes"
        bare = stt.transcribe(audio, None, bare_cfg)
        safe = stt.transcribe(audio, None, fw_cfg)
        row = {"clip": name, "gate": gate, "bare_words": len(bare["text"].split()),
               "firewall_words": len(safe["text"].split()), "bare_text": bare["text"][:100], "fw_text": safe["text"][:100]}
        gtxt = ""
        if name == "speech_with_gap" and gap:
            lo, hi = gap[0] + 1.0, gap[1] - 1.0
            row["gap_words_bare"], row["gap_words_fw"] = _words_in(bare, lo, hi), _words_in(safe, lo, hi)
            gtxt = f"{row['gap_words_bare']}/{row['gap_words_fw']}"
        rows.append(row)
        print(f"{name:22}{gate:>9}{row['bare_words']:>12}{row['firewall_words']:>16}{gtxt:>22}")
        if row["bare_text"]:
            print("    bare ->", row["bare_text"])
        if row["fw_text"]:
            print("    fw   ->", row["fw_text"])
    return rows

Overwriting eval/hallucination.py


In [ ]:
# 12. Load modules + GPU smoke test
import importlib, sys, time
import numpy as np

def reload_all():
    if "stages.stt" in sys.modules:
        try:
            sys.modules["stages.stt"].free_gpu()
        except Exception:
            pass
    importlib.invalidate_caches()
    for name in ["config", "errors", "stages.textutil", "stages.ingest", "stages.stt",
                 "stages.asr2", "pipeline_stt", "eval.run_grid", "eval.hallucination"]:
        if name in sys.modules:
            importlib.reload(sys.modules[name])

reload_all()
import config, errors
from config import CFG, SR, with_changes
from errors import PipelineError, ModelError
from stages import ingest, stt
import pipeline_stt
from eval import run_grid as G
from eval import hallucination as H

print(f"device={CFG.device} | model={CFG.model_size} | compute={CFG.compute_type}")
if CFG.device == "cpu":
    print("WARNING: no GPU found. Runtime -> Change runtime type -> T4 GPU, then restart and re-run from cell 2.")
t0 = time.time()
try:
    m = stt.get_model(CFG)
    segs, _ = m.transcribe(np.zeros(SR * 2, dtype="float32"), language="en")
    list(segs)
    print(f"OK: model loaded and ran a test inference in {time.time() - t0:.0f}s")
except Exception as e:
    print("FAILED:", type(e).__name__, str(e)[:300])
    print("-> Re-run cell 2; restart the runtime if you just installed packages;")
    print("   still failing? set WHISPER_DEVICE=cpu and WHISPER_MODEL=small in the environment and re-run this cell.")

device=cpu | model=large-v3 | compute=int8


OK: model loaded and ran a test inference in 116s


In [ ]:
# 13. Clips. Scripted test clip = read the script below aloud (about 75 s) and record it on your phone.
import shutil
from google.colab import files

REFERENCE_SCRIPT = """Okay everyone, let's start the weekly sync. First, the migration from MySQL to PostgreSQL. Priya says the staging database is ready, but we will not migrate production this quarter, because the load tests failed twice. Second, the CI/CD pipeline. We deploy through Kubernetes and Docker, and the build time is now fifteen minutes, down from fifty. Rahul suggested moving the API gateway to AWS, but nobody agreed, so we are keeping the current setup. Third, the launch. We decided the release date is the twenty third of October. Priya will update the Terraform scripts by Friday. Someone needs to write the rollback runbook. Rahul will send the benchmark report. That's all, thanks everyone."""
open(f"{CLIPS_DIR}/clipa.txt", "w", encoding="utf-8").write(REFERENCE_SCRIPT)

# Upload audio. Rules:
#  - scripted clip: name it clipa.<ext> (or set SCRIPTED_CLIP below to its uploaded file name)
#  - any other clip: upload <name>.<ext> AND <name>.txt (its reference transcript);
#    optional <name>.terms.txt = one important term per line (names, jargon) to check
#  - optional music clip for the hallucination test: music.wav (30 s)
UPLOAD = True
SCRIPTED_CLIP = None      # e.g. "WhatsApp Audio 2026-10-07 at 1.33.14 PM.m4a"
if UPLOAD:
    up = files.upload()
    for name in up:
        shutil.move(name, f"{CLIPS_DIR}/{name}")
if SCRIPTED_CLIP:
    ext = os.path.splitext(SCRIPTED_CLIP)[1]
    shutil.copy(f"{CLIPS_DIR}/{SCRIPTED_CLIP}", f"{CLIPS_DIR}/clipa{ext}")

pairs = G.find_pairs(CLIPS_DIR)
print("clips with a reference transcript:")
for a, r in pairs:
    print("  ", os.path.basename(a), "<->", os.path.basename(r))
if not pairs:
    print("none found: upload an audio file plus a .txt with the same name")

Saving clipa.m4a to clipa.m4a
clips with a reference transcript:
   clipa.m4a <-> clipa.txt


In [ ]:
# 14. Single run: transcript, metrics, flags, differences
def report_clip(audio_path, ref_path, glossary=None, cfg=None):
    ref = open(ref_path, encoding="utf-8").read()
    raw = pipeline_stt.process(audio_path, glossary=glossary, cfg=cfg,
                               on_progress=lambda s, p: print(f"\r{s:14s}{p:5.0%}", end=""))
    print("\n")
    terms = G.load_terms(audio_path)
    print("WER:", round(G.wer(ref, raw["text"]), 4), "| critical-token recall:", G.critical_recall(ref, raw["text"]))
    if terms:
        print("missing important terms:", G.missing_terms(raw["text"], terms) or "none")
    print("flags:", raw["flags"])
    print("warnings:", raw["warnings"] or "none")
    print("\n--- transcript (numbered lines, as Phase 3 will cite them) ---")
    print("\n".join(raw["lines"]))
    print("\n--- differences vs reference (after normalisation) ---")
    G.show_diff(ref, raw["text"])
    if raw["dropped"]:
        print("\n--- segments removed by the firewall ---")
        for d in raw["dropped"]:
            print(f"  {d['drop_reason']:32s} | {d['text']}")
    crit = [(s["id"], s["critical_low_conf"]) for s in raw["segments"] if s["critical_low_conf"]]
    if crit:
        print("\n--- numbers/negations/commitments the model was unsure about (do NOT auto-edit) ---")
        for sid, words in crit:
            print(f"  segment {sid}: {words}")
    return raw

audio_path, ref_path = pairs[0]
raw = report_clip(audio_path, ref_path, glossary=G.GLOSSARY)

cross-checking  88%

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

done           100%

WER: 0.0439 | critical-token recall: 0.917
missing important terms: none
flags: {'low_conf_words': 1, 'critical_low_conf': 1, 'dropped_segments': 0, 'unstable_segments': 0}
warnings: none

--- transcript (numbered lines, as Phase 3 will cite them) ---
[L001 00:00] Okay everyone, let's start the weekly sync. First, the migration from my SQL to PostgreSQL.
[L002 00:05] Priya says the staging database is ready, but we will not migrate production this quarter
[L003 00:11] because the load tests fail twice. Second, the CI-CD pipeline. We deploy through Kubernetes
[L004 00:18] and Docker and the build time is now 15 minutes, down from 50. Rahul suggested moving the API
[L005 00:24] API Gateway to AWS but nobody agreed, so we are keeping the current setup.
[L006 00:30] 3.
[L007 00:31] The launch.
[L008 00:32] We decided the release date is the 23rd of October.
[L009 00:36] Priya will update the Terraform scripts by Friday.
[L010 00:39] Someone needs to write the rollback 

In [ ]:
# 15. Hallucination test: bare Whisper vs the firewall on non-speech audio
speech = ingest.load_wav(ingest.convert_to_wav(audio_path))
clips = H.make_clips("outputs/halluc_clips", speech_audio=speech)
# Optional real music: put a 30 s clip named music.wav in your clips folder
music = f"{CLIPS_DIR}/music.wav"
extra = [ingest.convert_to_wav(music)] if os.path.exists(music) else []
hall_rows = H.run(clips, extra_wavs=extra)
import pandas as pd
pd.DataFrame(hall_rows).to_csv("outputs/hallucination_results.csv", index=False)
print("\nsaved outputs/hallucination_results.csv")
print("Reading it: 'bare words' > 0 on silence/noise = Whisper invented text; the firewall column should be 0.")
print("'gate' shows whether the Stage-0 speech check would reject that file outright.")

clip                       gate  bare words  firewall words   gap words (bare/fw)
silence                 BLOCKED           2               0                      
    bare -> Thank you.
white_noise             BLOCKED           2               0                      
    bare -> Thank you.
pink_noise              BLOCKED           2               0                      
    bare -> Thank you.
brown_noise             BLOCKED           2               0                      
    bare -> Thank you.
tones                   BLOCKED           2               0                      
    bare -> Thank you.
hum                     BLOCKED           2               0                      
    bare -> Thank you.
speech_with_gap          passes         115             113                   0/0
    bare -> Ok everyone, let's start the weekly sync. First the migration from my sql to postgre sql. Priya says
    fw   -> Okay everyone, let's start the weekly sync. First, the migration from my SQL to P

In [ ]:
# 16. Comparison grid: which settings are actually best on YOUR clips?
# ~13 configurations per clip; allow roughly (13 x clip length / 3) minutes. The last one downloads large-v3-turbo.
# To run only some: rows = G.run(pairs, include=["A", "B", "C", "H"])
rows = G.run(pairs)
summary = G.summarize(rows)
display(summary)
best = G.recommend(summary)
print("\nRecommended config:", best)
print("Rule: keep the default (B) unless another setting wins by >= 1 WER point without losing critical-token recall.")
print("With only a few clips, treat small differences as noise. To adopt a setting, edit its value in the config.py cell and re-run cell 12.")

{'clip': 'clipa.m4a', 'config': 'A bare whisper', 'WER': 0.0439, 'missing_terms': 0, 'crit_recall': 1.0, 'dropped': 0, 'seconds': 221.7, 'error': ''}
{'clip': 'clipa.m4a', 'config': 'B firewall (default)', 'WER': 0.0351, 'missing_terms': 0, 'crit_recall': 1.0, 'dropped': 0, 'seconds': 213.0, 'error': ''}
{'clip': 'clipa.m4a', 'config': 'C +glossary sentence', 'WER': 0.0439, 'missing_terms': 0, 'crit_recall': 0.917, 'dropped': 0, 'seconds': 213.9, 'error': ''}
{'clip': 'clipa.m4a', 'config': 'D +glossary list', 'WER': 0.0351, 'missing_terms': 0, 'crit_recall': 1.0, 'dropped': 0, 'seconds': 211.3, 'error': ''}
{'clip': 'clipa.m4a', 'config': 'E +hotwords only', 'WER': 0.0351, 'missing_terms': 0, 'crit_recall': 0.917, 'dropped': 0, 'seconds': 213.1, 'error': ''}
{'clip': 'clipa.m4a', 'config': 'F beam=1', 'WER': 0.0351, 'missing_terms': 0, 'crit_recall': 1.0, 'dropped': 0, 'seconds': 177.9, 'error': ''}
{'clip': 'clipa.m4a', 'config': 'G no_repeat_ngram=4', 'WER': 0.0351, 'missing_terms':

In [ ]:
# 17. OPTIONAL: second ASR (NVIDIA Parakeet TDT via onnx-asr) as an error locator. Whisper stays the primary STT.
cfg2 = with_changes(use_asr2=True)
raw2 = pipeline_stt.process(audio_path, glossary=G.GLOSSARY, cfg=cfg2)
a2 = raw2.get("asr2")
if not a2:
    print("Cross-check not available:", raw2["warnings"])
else:
    print(f"disagreement rate: {a2['disagreement_rate']:.1%} | critical disagreements: {a2['critical_disagreements']} "
          f"| segments the 2nd model does not support: {a2['unsupported_segments']} | took {a2['seconds']}s")
    for d in a2["disagreements"][:25]:
        print(f"  [{d['start']:6.1f}s] {d['type']:8s} whisper={d['whisper']!r:28} parakeet={d['parakeet']!r:28} critical={d['critical']}")
    ref2 = open(ref_path, encoding="utf-8").read()
    print("\nDo disagreements point at Whisper's REAL errors? ->", G.error_capture(ref2, raw2))
    print("(recall = share of Whisper's errors that were flagged; precision = share of flags that were real errors)")
    print("Ship this only if recall is clearly useful across several clips; otherwise report it as a negative result.")

In [ ]:
# 18. Export: version pins, outputs, zip for your teammate
import subprocess, zipfile
print(subprocess.run("pip freeze | grep -iE '^(faster-whisper|ctranslate2|av|onnx-asr|onnxruntime|jiwer|num2words|nvidia-cudnn-cu12|nvidia-cublas-cu12)=='",
                     shell=True, capture_output=True, text=True).stdout)
print(pipeline_stt.save_outputs(raw, "outputs"))
with zipfile.ZipFile("meeting_assistant_phase1.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for top in ("config.py", "errors.py", "pipeline_stt.py"):
        z.write(top)
    for folder in ("stages", "eval", "outputs"):
        for root, _, fs in os.walk(folder):
            if "__pycache__" in root:
                continue
            for f in fs:
                z.write(os.path.join(root, f))
from google.colab import files
files.download("meeting_assistant_phase1.zip")